# Prepare Data

In [1]:
import pandas as pd

In [2]:
dataset = pd.read_csv("/kaggle/input/notebooks/davidvista/pinyin-eval-dataset-labelling/eval-chinese-short-sentences-pinyin.csv")

In [3]:
dataset.head()

,text,pinyin
0,他赢了布什说我很高兴是他赢了,ta1 ying2 le bu4 shen2 shuo1 wo3 hen3 gao1 xin...
1,你根本想不到有多畅销赛义德说,ni3 gen1 ben3 xiang3 bu4 dao4 you3 duo1 chang4...
2,国家主席是个虚职实权掌握在共产党总书记的手里,guo2 jia1 zhu3 xi2 shi4 ge4 xu1 zhi2 shi2 quan...
3,如果你批评以色列政府有些人会因此说你是反犹主义者,ru2 guo3 ni3 pi1 ping2 yi3 se4 lie4 zheng4 fu3...
4,我是一十六号下次请还要叫我啊,wo3 shi4 yi1 shi2 liu4 hao4 xia4 ci4 qing3 hai...


# Count Frequencies

In [4]:
from collections import Counter, defaultdict
from tqdm import tqdm

char_pinyin_freq = Counter()

for _, row in tqdm(dataset.iterrows(), total=len(dataset)):

    text = row["text"]
    pinyins = row["pinyin"].split()

    if len(text) != len(pinyins):
        continue

    for char, py in zip(text, pinyins):

        if char.isspace():
            continue

        char_pinyin_freq[(char, py)] += 1

100%|██████████| 181302/181302 [00:14<00:00, 12608.89it/s]


In [5]:
pinyin_groups = defaultdict(list)

for (char, py), freq in char_pinyin_freq.items():

    pinyin_groups[py].append(
        (char, freq)
    )

In [6]:
for py in pinyin_groups:

    pinyin_groups[py] = sorted(
        pinyin_groups[py],
        key=lambda x: x[1],
        reverse=True
    )

In [7]:
MIN_CANDIDATES = 2
MAX_CANDIDATES = 10

homophone_inventory = {}

for py, chars in pinyin_groups.items():

    if len(chars) < MIN_CANDIDATES:
        continue

    homophone_inventory[py] = [
        char
        for char, freq in chars[:MAX_CANDIDATES]
    ]

# Extracting Homophones

In [8]:
exact_homophones = {}

for py, chars in homophone_inventory.items():

    for char in chars:

        exact_homophones[char] = [
            c
            for c in chars
            if c != char
        ]

In [9]:
MIN_FREQ = 50
MAX_CANDIDATES = 10

homophone_inventory = {}

for py, chars in pinyin_groups.items():

    filtered = [
        (char, freq)
        for char, freq in chars
        if freq >= MIN_FREQ
    ]

    if len(filtered) >= 2:

        homophone_inventory[py] = {
            "chars": [char for char, freq in filtered[:MAX_CANDIDATES]],
            "freqs": [freq for char, freq in filtered[:MAX_CANDIDATES]]
        }

In [10]:
homophone_inventory["zai4"]

{'chars': ['在', '再', '载', '產'], 'freqs': [76178, 3514, 434, 76]}

In [11]:
import pickle

with open("homophone_inventory.pkl", "wb") as f:
    pickle.dump(homophone_inventory, f)